In [100]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import VBox, HBox, FloatSlider, Output, Dropdown, FloatLogSlider
from IPython.display import display, clear_output

%matplotlib inline

In [118]:
plt.close('all')

c = 344
n = 128
fig_width = 13
fov = np.pi / 2

w_mics = FloatSlider(
    value=1, min=1, max=8,
    step=1,
    description="Mics", continuous_update=True, readout_format=".0f"
)
w_indent = FloatSlider(
    value=1.0, min=0.0, max=1,
    step=0.001,
    description="Indent", continuous_update=True, readout_format=".3f"
)
w_f = FloatLogSlider(
    value=1000, base = 10, min=2, max=np.log10(4000),step=0.001,
    description="Frequency", continuous_update=True, readout_format=".1f"
)
selectionTypes = ["even/odd", "opposite sides", "random"]
w_selection = Dropdown(options=selectionTypes, value="even/odd", description="Selection")

w_mics.layout.width = "450px"
w_indent.layout.width = "700px"
w_f.layout.width = "450px"
w_selection.layout.width = "200px"

def squarePattern(edgeLength, micsPerEdge, oddIndent = 1):
    # place microphones at the edges
    pattern = np.array([i*edgeLength/micsPerEdge + 0j for i in range(micsPerEdge)])
    pattern -= (1 + 1j) * edgeLength / 2
    pattern = np.concat([pattern, pattern * 1j, -pattern, -1j * pattern])
    pattern = np.array([pattern.real, pattern.imag])
    pattern[:,[2*i+1 for i in range(2*micsPerEdge)]] *= oddIndent
    return pattern

out = Output()
def update_plot(change):
    fc = w_f.value
    micsPerEdge = int(w_mics.value)
    pattern = squarePattern(0.5, micsPerEdge, w_indent.value)
    patternX, patternY = pattern
    pattern = pattern.transpose()

    k = 2 * np.pi * fc / c

    x = np.linspace(-fov, fov, n)
    y = np.linspace(-fov, fov, n)

    todeg = 1 / np.pi * 180
    xv, yv = np.meshgrid(x * todeg, y * todeg)
    p = np.stack(np.meshgrid(k * np.sin(x), k * np.sin(y)), axis=-1)

    def computeBeamPattern(micPositions):
        mp = micPositions
        return np.sum(np.array([np.exp(np.dot(p, mp[i])*1j) for i in range(len(mp))]), axis=0) / len(mp)

    def plotMesh(ax, z):
        c = ax.pcolormesh(xv, yv, np.abs(z), shading='auto', cmap='viridis')
        plt.colorbar(c, ax=ax, label='Gain (linear)')

    idx1 = []
    if(w_selection.value == selectionTypes[0]):
         idx1 = [2*i for i in range(2*micsPerEdge)]
    if(w_selection.value == selectionTypes[1]):
         idx1 = np.array([(i + 2 * micsPerEdge, i) for i in range(len(pattern)//4)]).flatten()
    if(w_selection.value == selectionTypes[2]):
         idx1 = np.round(np.random.rand(4*micsPerEdge)).astype(bool)
         print(idx1)
    
    idx2 = np.ones(len(pattern), dtype=bool)
    idx2[idx1] = False
    subPattern1 = pattern[idx1]
    subPattern2 = pattern[idx2]

    with out:
        clear_output(wait=True)
        fig, ((ax1, ax2, ax3), (ax4, ax5, ax6)) = plt.subplots(
            2, 3, figsize=(fig_width, fig_width * 0.6),
            constrained_layout=True
        )
        ax1.scatter(patternX, patternY, s=20, c='C0')
        ax1.scatter(patternX[idx1], patternY[idx1], s=20, c='C1')
        z0 = computeBeamPattern(pattern)
        z1 = computeBeamPattern(subPattern1)
        z2 = computeBeamPattern(subPattern2)
        z3 = np.maximum(0, np.real(2 * z1 * np.conj(z2)) / np.real(z1 * np.conj(z1) + z2 * np.conj(z2) + 1e-18))
        plotMesh(ax2, z0)
        plotMesh(ax3, z0 * z3)
        plotMesh(ax6, z3)
        plotMesh(ax4, z1)
        plotMesh(ax5, z2)
        ax2.set_title('Delay and sum beam pattern')
        ax3.set_title('Cropac beam pattern')
        ax6.set_title('Cropac gain pattern')
        ax4.set_title('Subset beam pattern 1')
        ax5.set_title('Subset beam pattern 2')
        plt.show()

for w in (w_f, w_mics, w_indent, w_selection):
        w.observe(update_plot, names="value")

display(VBox([
    HBox([w_f, w_mics]),
    HBox([w_indent, w_selection]),
    out
]))

update_plot(None)